In [ ]:
# Set up before running plots
# Before I began I downloaded "sudo pip3 install plotly" on my mac

In [ ]:
# import libraries
import pandas as pd
import plotly.express as px
import plotly.io as pio

In [ ]:
#Directory
import os
os.chdir('/Users/tasia/Desktop/Data Analysis/Assessment')

In [ ]:
# load data
df_gdp = pd.read_csv('cross_sectional_gdp_2023.csv')
df_unp  = pd.read_csv('cross_sectional_unemployment_2023.csv')

# eyeball
print(df_gdp.head())
print(df_unp.head())

In [ ]:
# convert missing values "..." to numeric
df_gdp['2023 [YR2023]'] = pd.to_numeric(df_gdp['2023 [YR2023]'], errors='coerce')

df_unp['2023 [YR2023]'] = pd.to_numeric(df_unp['2023 [YR2023]'], errors='coerce')

In [ ]:
# merge dataframes (inner join!)
df = pd.merge(left = df_gdp, 
              right = df_unp,
              how = 'inner',
              left_on = ['Country Code'],
              right_on = ['Country Code'],
              )
# rename columns
df = df.rename(columns={
    '2023 [YR2023]_x': 'GDP per capita (2023)',
    '2023 [YR2023]_y': 'Unemployment rate (2023)'
})

# eyeball
df.head()

In [ ]:
#adding region information
import requests

url = "https://api.worldbank.org/v2/country?format=json&per_page=400"

response = requests.get(url)
response.raise_for_status()
countries = response.json()[1]

#dataframe with country codes and regions
regions = pd.DataFrame([{"Country Code": country["id"],
                         "Region": country["region"]["value"]}
                        for country in countries])

#merging region information with GDP and unemployment data
df = pd.merge(df, regions, on="Country Code", how="left")

#removing missing values and World Bank aggregates
df_regions = df.dropna(subset=["GDP per capita (2023)","Unemployment rate (2023)","Region"])

df_regions = df_regions[df_regions["Region"] != "Aggregates"]

#list available regions
regions_list = sorted(df_regions["Region"].unique())

# check the result
df[["Country Name_x", "Region"]].head()

In [ ]:
# display summary statistics
df.describe()

In [ ]:
# plotting a GDP per capita bar chart for EU15 countries in 2023
fig = px.bar(df[(df["Country Name_x"].isin(["Austria",
                                     "Belgium",
                                     "Denmark",
                                     "Finland",
                                     "France",
                                     "Germany",
                                     "Greece",
                                     "Ireland",
                                     "Italy",
                                     "Luxembourg",
                                     "Netherlands",
                                     "Portugal",
                                     "Spain",
                                     "Sweden",
                                     "United Kingdom"]))], 
              x = "Country Name_x", 
              y = "GDP per capita (2023)",
              title = "GDP per capita across EU15 countries in 2023")
fig.show()

In [ ]:
#plotting an interactive GDP per capita bar chart by European region
#selecting the EU15 countries
eu15 = ["Austria", "Belgium", "Denmark", "Finland", "France", "Germany", "Greece", "Ireland", "Italy", "Luxembourg", "Netherlands", 
        "Portugal", "Spain", "Sweden", "United Kingdom"]

df_eu15 = df[df["Country Name_x"].isin(eu15)].copy()

#group countries into European regions
regions = {"Northern Europe": ["Denmark", "Finland", "Ireland", "Sweden", "United Kingdom"],
           "Western Europe": ["Austria", "Belgium", "France", "Germany", "Luxembourg", "Netherlands"],
           "Southern Europe": ["Greece", "Italy", "Portugal", "Spain"]}

#creating a region column
for region, countries in regions.items():
    df_eu15.loc[
        df_eu15["Country Name_x"].isin(countries), "European Region"
    ] = region

#creating the bar chart
fig = px.bar(
    df_eu15,
    x="Country Name_x",
    y="GDP per capita (2023)",
    title="GDP per capita across EU15 countries by European region in 2023")

#creating the dropdown menu
buttons = [dict(label="All EU15 countries",method="update",
                args=[{"x": [df_eu15["Country Name_x"].tolist()],
                       "y": [df_eu15["GDP per capita (2023)"].tolist()]}])]

for region in regions:
    region_data = df_eu15[df_eu15["European Region"] == region]

    buttons.append(dict(label=region,method="update",
                        args=[{"x": [region_data["Country Name_x"].tolist()],
                               "y": [region_data["GDP per capita (2023)"].tolist()]}]))

#positioning the dropdown menu above the chart
fig.update_layout(updatemenus=[dict(
    buttons=buttons,
    direction="down",
    showactive=True,
    x=0,
    y=1.35,
    xanchor="left",
    yanchor="top")],
                  margin=dict(t=180),
                  title=dict(x=0.5,xanchor="center",y=0.90))

fig.show()

In [ ]:
#plotting an Unemployment Rate bar chart for EU15 countries in 2023
fig = px.bar(df[(df["Country Name_y"].isin(["Austria",
                                     "Belgium",
                                     "Denmark",
                                     "Finland",
                                     "France",
                                     "Germany",
                                     "Greece",
                                     "Ireland",
                                     "Italy",
                                     "Luxembourg",
                                     "Netherlands",
                                     "Portugal",
                                     "Spain",
                                     "Sweden",
                                     "United Kingdom"]))], 
              x = "Country Name_y", 
              y = "Unemployment rate (2023)",
              title = "Unemployment rate across EU15 countries in 2023")
fig.show()

In [ ]:
#plotting an interactive Unemployment Rate bar chart for EU15 countries in 2023
#select EU15 countries
eu15 = ["Austria", "Belgium", "Denmark", "Finland", "France", "Germany", "Greece", "Ireland", "Italy", "Luxembourg", "Netherlands", 
        "Portugal", "Spain", "Sweden", "United Kingdom"]

df_eu15 = df[df["Country Name_y"].isin(eu15)].copy()

#group countries into European regions
regions = {"Northern Europe": ["Denmark", "Finland", "Ireland", "Sweden", "United Kingdom"],
           "Western Europe": ["Austria", "Belgium", "France", "Germany", "Luxembourg", "Netherlands"],
           "Southern Europe": ["Greece", "Italy", "Portugal", "Spain"]}

#create a region column
for region, countries in regions.items():
    df_eu15.loc[df_eu15["Country Name_y"].isin(countries), "European Region"] = region

#create the bar chart
fig = px.bar(df_eu15,
    x="Country Name_y",
    y="Unemployment rate (2023)",
    title="Unemployment rates across EU15 countries by European region in 2023")

#create dropdown menu
buttons = [dict(
        label="All EU15 countries",
        method="update",
        args=[{"x": [df_eu15["Country Name_y"].tolist()],
               "y": [df_eu15["Unemployment rate (2023)"].tolist()]}])]

for region in regions:
    region_data = df_eu15[df_eu15["European Region"] == region]

    buttons.append(dict(label=region,
            method="update",
            args=[{"x": [region_data["Country Name_y"].tolist()],
                   "y": [region_data["Unemployment rate (2023)"].tolist()]}]))

#add dropdown menu above the chart
fig.update_layout(updatemenus=[dict(buttons=buttons,
                                    direction="down",
                                    showactive=True,
                                    x=0,
                                    y=1.35,
                                    xanchor="left",
                                    yanchor="top")],
                  margin=dict(t=180),
                  title=dict(
                      x=0.5,
                      xanchor="center",
                      y=0.90),yaxis_title="Unemployment rate (%)")
fig.show()

In [ ]:
# plotting a clustered bar chart by World Bank region in 2023

# calculating average GDP per capita
average_gdp = df_regions["GDP per capita (2023)"].mean()

# grouping countries by GDP per capita
df_clustered = df_regions.copy()

df_clustered["GDP Group"] = "Below-average GDP"
df_clustered.loc[df_clustered["GDP per capita (2023)"] >= average_gdp, "GDP Group"] = "Above-average GDP"

# calculating average unemployment by region and GDP group
df_clustered = df_clustered.groupby(["Region", "GDP Group"], as_index = False)["Unemployment rate (2023)"].mean()

# plotting clustered bar chart
fig = px.bar(df_clustered,
             x = "Region",
             y = "Unemployment rate (2023)",
             color = "GDP Group",
             barmode = "group",
             title = "Average unemployment by World Bank region and GDP group in 2023")

fig.update_layout(yaxis_title = "Average unemployment rate (%)")

fig.show()

In [ ]:
#plotting an interactive clustered bar chart by World Bank region

fig = px.bar(df_clustered,
             x = "Region",
             y = "Unemployment rate (2023)",
             color = "GDP Group",
             barmode = "group",
             title = "Average unemployment by World Bank region and GDP group in 2023")

# creating dropdown buttons
regions_list = sorted(df_clustered["Region"].unique())

buttons = [dict(label = "All regions",
                method = "update",
                args = [{"x": [df_clustered[df_clustered["GDP Group"] == trace.name]["Region"].tolist() for trace in fig.data],
                         "y": [df_clustered[df_clustered["GDP Group"] == trace.name]["Unemployment rate (2023)"].tolist() for trace in fig.data]}])]

for region in regions_list:
    region_data = df_clustered[df_clustered["Region"] == region]

    buttons.append(dict(label = region,
                        method = "update",
                        args = [{"x": [region_data[region_data["GDP Group"] == trace.name]["Region"].tolist() for trace in fig.data],
                                 "y": [region_data[region_data["GDP Group"] == trace.name]["Unemployment rate (2023)"].tolist() for trace in fig.data]}]))

# adding dropdown menu
fig.update_layout(updatemenus = [dict(buttons = buttons,
                        direction = "down",
                        showactive = True,
                        x = 0,
                        y = 1.35,
                        xanchor = "left",
                        yanchor = "top")],
    margin = dict(t = 180),
    title = dict(x = 0.5,
                 xanchor = "center",
                 y = 0.90),
    yaxis_title = "Average unemployment rate (%)")

fig.show()

In [ ]:
# plot a scatter plot of GDP per capita and unemployment in 2023
fig = px.scatter(df, 
                 x = "GDP per capita (2023)", 
                 y = "Unemployment rate (2023)",
                 hover_data = ["Country Name_x"],
                 title = "GDP per capita and unemployment rates in 2023")
fig.show()

In [ ]:
#plotting an interactive scatter plot by region
#creating the interactive scatter plot with region selection
fig = px.scatter(
    df_regions,
    x="GDP per capita (2023)",
    y="Unemployment rate (2023)",
    color="Region",
    hover_name="Country Name_x",
    title="GDP per capita and unemployment by region in 2023")

#creating dropdown buttons for each region
regions_list = sorted(df_regions["Region"].unique())

buttons = [dict(
        label="All regions",
        method="update",
        args=[{"visible": [True] * len(fig.data)}])]

for region in regions_list:
    buttons.append(dict(label=region,
            method="update",
            args=[{"visible": [trace.name == region for trace in fig.data]}]))


# adding dropdown menu above the scatter plot
fig.update_layout(updatemenus=[dict(
    buttons=buttons,
    direction="down",
    showactive=True,
    x=0,
    y=1.35,
    xanchor="left",
    yanchor="top")],
                  margin=dict(t=180),
                  title=dict(
                      x=0.5,
                      xanchor="center",
                      y=0.90))
fig.show()

In [ ]:
# plot a bivariate histogram (a heatmap)
fig = px.density_heatmap(df, 
                         x = "GDP per capita (2023)", 
                         y = "Unemployment rate (2023)",
                         nbinsx = 30, 
                         nbinsy = 30)
fig.show()

In [ ]:
#plotting an interactive bivariate histogram (heatmap) by region
# create a heatmap for all regions
fig = px.density_heatmap(df_regions,
                         x="GDP per capita (2023)",
                         y="Unemployment rate (2023)",
                         nbinsx=30,
                         nbinsy=30,
                         title="GDP per capita and unemployment by region in 2023")

#creating dropdown buttons
buttons = [dict(label="All regions",method="update",args=[{"x": [df_regions["GDP per capita (2023)"].tolist()],
                                                           "y": [df_regions["Unemployment rate (2023)"].tolist()]}])]

for region in regions_list:
    region_data = df_regions[df_regions["Region"] == region]

    buttons.append(dict(label=region,method="update",args=[{"x": [region_data["GDP per capita (2023)"].tolist()],
                                                            "y": [region_data["Unemployment rate (2023)"].tolist()]}]))
    
#add dropdown menu above the heatmap
fig.update_layout(updatemenus=[dict(buttons=buttons,
                                    direction="down",
                                    showactive=True,
                                    x=0,
                                    y=1.35,
                                    xanchor="left",
                                    yanchor="top")],
                  margin=dict(t=180),
                  title=dict(
                      x=0.5,
                      xanchor="center",
                      y=0.90))

fig.show()

In [ ]:
# plot a heatmap with marginal histograms
fig = px.density_heatmap(df, 
                         x = "GDP per capita (2023)", 
                         y = "Unemployment rate (2023)",
                         marginal_x = "histogram", 
                         marginal_y = "histogram",
                         nbinsx = 30, 
                         nbinsy = 30)
fig.show()

In [ ]:
#plotting an interactive heatmap with marginal histograms by region
#use the same valid country observations for every selection
df_heatmap = df_regions.copy()

#create the interactive heatmap
fig = px.density_heatmap(df_heatmap,
                         x="GDP per capita (2023)",
                         y="Unemployment rate (2023)",
                         marginal_x="histogram",
                         marginal_y="histogram",
                         nbinsx=30,
                         nbinsy=30,
                         title="GDP per capita and unemployment with marginal histograms by region in 2023")

#create dropdown options
regions_list = sorted(df_heatmap["Region"].dropna().unique())

#heatmap, top histogram and right histogram

def make_region_update(data):
    gdp = data["GDP per capita (2023)"].tolist()
    unemployment = data["Unemployment rate (2023)"].tolist()

    return {"x": [gdp, gdp, None], "y": [unemployment, None, unemployment]}

buttons = [dict(label="All regions", method="update", args=[make_region_update(df_heatmap)])]

for region in regions_list:
    region_data = df_heatmap[df_heatmap["Region"] == region]
    
    buttons.append(dict(label=region,
                        method="update",
                        args=[make_region_update(region_data)]))

#add dropdown above the chart
fig.update_layout(updatemenus=[dict(buttons=buttons,
                                    direction="down",
                                    showactive=True,
                                    x=0,
                                    y=1.35,
                                    xanchor="left",
                                    yanchor="top")],
                  margin=dict(t=180),
                  title=dict(x=0.5,
                      xanchor="center",
                             y=0.90))

fig.show()

In [ ]:
#download population data for 2023
import requests

url = "https://api.worldbank.org/v2/country/all/indicator/SP.POP.TOTL?format=json&date=2023&per_page=500"
data = requests.get(url).json()[1]

# create dataframe
df_population = pd.DataFrame(data)
df_population["Country Code"] = df_population["countryiso3code"]
df_population["Population (2023)"] = df_population["value"]

# merge dataframes (inner join!)
df_bubble = pd.merge(left = df_regions,
                     right = df_population[["Country Code", "Population (2023)"]],
                     how = "inner",
                     on = "Country Code")

# remove missing population values
df_bubble = df_bubble.dropna(subset = ["Population (2023)"])

# eyeball
df_bubble.head()

In [ ]:
#plotting an interactive bubble chart by region
fig = px.scatter(df_bubble,
                 x = "GDP per capita (2023)",
                 y = "Unemployment rate (2023)",
                 size = "Population (2023)",
                 color = "Region",
                 hover_name = "Country Name_x",
                 size_max = 55,
                 title = "GDP per capita, unemployment and population by region in 2023")

#creating dropdown buttons for each region
regions_list = sorted(df_bubble["Region"].unique())

buttons = [dict(label = "All regions",
                method = "update",
                args = [{"visible": [True] * len(fig.data)}])]

for region in regions_list:
    buttons.append(dict(label = region,
                        method = "update",
                        args = [{"visible": [trace.name == region for trace in fig.data]}]))

#adding dropdown menu
fig.update_layout(updatemenus = [dict(buttons = buttons,
                        direction = "down",
                        showactive = True,
                        x = 0,
                        y = 1.35,
                        xanchor = "left",
                        yanchor = "top")],
    margin = dict(t = 180),
    title = dict(x = 0.5,
                 xanchor = "center",
                 y = 0.90))

fig.show()